# Glover Bound Tightness

This notebook demonstrates the **Glover (1984) certified approximation bound** for SSM truncation:

$$\|H - H_k\|_H \leq 2\sigma_{k+1}$$

where $H_k$ is the transfer function after keeping only the top-$k$ balanced modes, and $\sigma_{k+1}$ is the $(k+1)$-th Hankel singular value.

**What we show:**
1. The bound holds for all k (never violated)
2. The bound is tight: the ratio (actual error) / (bound) is roughly constant across k
3. The HSV spectrum predicts which modes can be safely truncated
4. Truncation error vs k for different task complexities

**Reference:** Glover (1984), Theorem 2; §2.3 of the paper; `ssi.glover_bound`.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

import ssm_interpret as ssi

print(f'ssm_interpret {ssi.__version__}')

In [ ]:
# We use a larger model (d=8) to have more modes to truncate across
T_SEQ = 64
M = 4
D_STATE = 8  # overcomplete: more modes than channels
RHOS = [0.99, 0.92, 0.75, 0.40]
EPOCHS = 25000

def make_ema_data(n, T, rhos, seed=0):
    rng = np.random.default_rng(seed)
    u = rng.uniform(-1, 1, (n, T, len(rhos))).astype(np.float32)
    y = np.zeros_like(u)
    for k, rho in enumerate(rhos):
        s = 0.0
        for t in range(T):
            s = rho * s + (1 - rho) * u[:, t, k]
            y[:, t, k] = s
    return torch.tensor(u), torch.tensor(y)

class LinearSSM(nn.Module):
    def __init__(self, m, d, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.A_raw = nn.Parameter(torch.randn(d, d) * 0.1)
        self.B = nn.Parameter(torch.randn(d, m) * 0.1)
        self.C = nn.Parameter(torch.randn(m, d) * 0.1)
        self.d = d

    def _A(self):
        sr = torch.linalg.matrix_norm(self.A_raw, ord=2)
        return self.A_raw / sr.clamp(min=1.0) * 0.99

    def forward(self, u):
        B_, T_, _ = u.shape
        A = self._A()
        x = torch.zeros(B_, self.d)
        ys = []
        for t in range(T_):
            x = x @ A.T + u[:, t, :] @ self.B.T
            ys.append(x @ self.C.T)
        return torch.stack(ys, dim=1)

    def get_matrices(self):
        with torch.no_grad():
            return (self._A().numpy().astype(np.float64),
                    self.B.detach().numpy().astype(np.float64),
                    self.C.detach().numpy().astype(np.float64))

u_tr, y_tr = make_ema_data(3000, T_SEQ, RHOS)
u_te, y_te = make_ema_data(512, T_SEQ, RHOS, seed=1)

print(f'Training d={D_STATE} model on {M}-channel EMA...')
model = LinearSSM(M, D_STATE, seed=0)
opt = optim.Adam(model.parameters(), lr=3e-3)
sched = optim.lr_scheduler.MultiStepLR(opt, milestones=[10000, 20000], gamma=0.3)
for ep in range(EPOCHS):
    opt.zero_grad()
    F.mse_loss(model(u_tr), y_tr).backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    sched.step()
    if (ep + 1) % 5000 == 0:
        with torch.no_grad():
            print(f'  epoch {ep+1}: loss={F.mse_loss(model(u_tr), y_tr).item():.4e}')

A, B, C = model.get_matrices()
Ab, Bb, Cb, T_bal, sigma = ssi.balanced_realization(A, B, C, T=T_SEQ)
sigma_np = np.array(sigma)
print(f'\nHSVs: {sigma_np.round(4)}')

## 1. Bound vs Actual Error for Each Truncation Level

In [ ]:
def hankel_norm(H):
    """Approximate Hankel norm via largest singular value of the impulse response matrix."""
    H_np = np.array(H)  # (p, T, m)
    p, T_, m = H_np.shape
    # Stack H as a block-Hankel-like matrix
    H_mat = H_np.reshape(p * T_, m)
    return float(np.linalg.norm(H_mat, ord=2))  # spectral norm as proxy

# Full impulse response
H_full = ssi.impulse_response(Ab, Bb, Cb, T=T_SEQ)

k_vals = list(range(1, D_STATE))
actual_errors = []
glover_bounds = []
ratios = []

for k in k_vals:
    Ab_k, Bb_k, Cb_k = ssi.truncate(Ab, Bb, Cb, k=k)
    H_k = ssi.impulse_response(Ab_k, Bb_k, Cb_k, T=T_SEQ)

    # Pad H_k to match H_full dimensions for subtraction
    H_k_np = np.array(H_k)
    H_full_np = np.array(H_full)
    err = hankel_norm(H_full_np - H_k_np)
    bound = ssi.glover_bound(sigma, k=k)

    actual_errors.append(err)
    glover_bounds.append(bound)
    ratios.append(err / (bound + 1e-12))
    print(f'k={k}: actual_err={err:.4f}  bound={bound:.4f}  ratio={err/(bound+1e-12):.3f}  {'OK' if err <= bound else 'VIOLATED'}')

print(f'\nBound hold rate: {sum(a<=b for a,b in zip(actual_errors, glover_bounds))}/{len(k_vals)}')
print(f'Mean ratio (actual/bound): {np.mean(ratios):.3f} (should be < 1.0)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.semilogy(k_vals, actual_errors, 'o-', color='#2196F3', label='Actual ||H - H_k||', linewidth=2)
ax.semilogy(k_vals, glover_bounds, 's--', color='#F44336', label='Glover bound 2σ_{k+1}', linewidth=2)
ax.fill_between(k_vals, actual_errors, glover_bounds, alpha=0.15, color='#F44336',
                label='Slack (bound - actual)')
ax.set_xlabel('Number of modes retained (k)')
ax.set_ylabel('Approximation error (log scale)')
ax.set_title('Glover Bound: Certified vs Actual Error')
ax.legend()
ax.set_xticks(k_vals)

ax = axes[1]
ax.bar(k_vals, ratios, color='#FF9800', edgecolor='k', linewidth=0.8)
ax.axhline(1.0, color='#F44336', linestyle='--', linewidth=1.5, label='Bound = actual (ratio=1)')
ax.set_xlabel('Number of modes retained (k)')
ax.set_ylabel('Actual error / Glover bound')
ax.set_title('Bound Tightness (ratio < 1 = valid)')
ax.set_xticks(k_vals)
ax.legend()
ax.set_ylim(0, 1.2)

plt.tight_layout()
plt.savefig('glover_bound.pdf', bbox_inches='tight')
plt.show()
print('Fig saved: glover_bound.pdf')

## 2. HSV-Based Model Compression

In [ ]:
# Find the minimal k such that the approximation error is below a tolerance
tols = [0.5, 0.1, 0.05, 0.01]
print('Certified compression: minimum k for given error tolerance')
print(f'{"tolerance":<12} {"min k (bound)":<16} {"min k (actual)":<16} {"compression"}')
print('-' * 60)

for tol in tols:
    # From bound: need 2*sigma[k] <= tol => sigma[k] <= tol/2
    k_bound = next((k+1 for k, s in enumerate(sigma_np) if s <= tol/2), D_STATE)
    # From actual measurement
    k_actual = next((k for k, e in zip(k_vals, actual_errors) if e <= tol), D_STATE)
    print(f'{tol:<12.3f} {k_bound:<16} {k_actual:<16} {D_STATE}/{k_bound}={D_STATE/max(k_bound,1):.1f}×')

print()
print('Interpretation: the bound tells you the safe compression level without measuring actual error.')
print('For this model, the HSV gap around k=4 indicates strong compression is possible.')

In [ ]:
# HSV spectrum with compression regions highlighted
fig, ax = plt.subplots(figsize=(8, 4))

bars = ax.bar(range(D_STATE), sigma_np,
              color=['#2196F3' if i < M else '#BDBDBD' for i in range(D_STATE)],
              edgecolor='k', linewidth=0.8)
ax.axhline(0.1/2, color='#4CAF50', linestyle='--', linewidth=1.5, label='Glover threshold for tol=0.1')
ax.axhline(0.05/2, color='#FF9800', linestyle='--', linewidth=1.5, label='Glover threshold for tol=0.05')
ax.set_xlabel('Mode index k')
ax.set_ylabel('Hankel singular value σ_k')
ax.set_title('HSV Spectrum: Blue = live circuits, Gray = safely truncatable')
ax.legend()
ax.set_xticks(range(D_STATE))
for k, s in enumerate(sigma_np):
    ax.text(k, s + 0.002, f'{s:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('hsv_compression.pdf', bbox_inches='tight')
plt.show()
print('Fig saved: hsv_compression.pdf')